# DefectosCafeVerde grouped — RATF1 seed 42

Fresh 50-epoch Redundancy-Aware Selective Texture Fusion from official YOLO26n. RGB and the native box path remain intact; test is never extracted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, hashlib, importlib, io, json, os, shutil, subprocess, sys, tarfile, time
ARM='RATF1'; BRANCH='codex/defectoscafeverde-ratf1'
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_ARCHIVE_BYTES=686474752
EXPECTED_ARCHIVE_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
drive_roots=[root for root in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if root.exists()]
archives=sorted({p for root in drive_roots for p in root.rglob(ARCHIVE_NAME) if p.is_file()})
valid_archives=[p for p in archives if p.stat().st_size==EXPECTED_ARCHIVE_BYTES and sha256(p)==EXPECTED_ARCHIVE_SHA]
complete_parts=[]
for scheme,count in (('part16',41),('chunk',11),('part',2)):
    by_parent={}
    for root in drive_roots:
        for p in root.rglob(f'{ARCHIVE_NAME}.{scheme}-*'):
            if p.is_file(): by_parent.setdefault(str(p.parent),{})[p.name]=p
    expected=[f'{ARCHIVE_NAME}.{scheme}-{i:03d}' for i in range(count)]
    for parent,rows in by_parent.items():
        if sorted(rows)==expected and sum(rows[n].stat().st_size for n in expected)==EXPECTED_ARCHIVE_BYTES: complete_parts.append((scheme,parent,[rows[n] for n in expected]))
ARCHIVE_ANCHOR=None
if valid_archives:
    ARCHIVE=valid_archives[0]; ARCHIVE_ANCHOR=ARCHIVE; ARCHIVE_SOURCE='single archive'
elif complete_parts:
    scheme,parent,parts=complete_parts[0]; ARCHIVE=WORK/ARCHIVE_NAME; ARCHIVE_ANCHOR=parts[0]; ARCHIVE_SOURCE=f'{len(parts)} {scheme} parts dari {parent}'
    with ARCHIVE.open('wb') as target:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError('SHA hasil gabung tidak cocok')
else:
    print('Bundle tidak terlihat lewat drive.mount(); memakai fallback Drive API.',flush=True)
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    chunk_ids=['1pAwWlkABja1Epaw1U8Uxre2JorfXwE3P','1UoZkqJbHstFcA9WeXW0KNeisGasqkci6','1kY_xwFOHW17eQQ7x1TDgM6F4ySRyg2gD','1HzW5oJnMOoO7HwWY4Ic-dK8SE-3uy1rH','1ydHf2EqZIJiA_I0udSL68QzbdscRvKtu','1ip1VVRQ6n1NW7Zr5DCU3iajy54Zgsd2B','10cobK2wuX_VsqOd8DfSiwHZcfoXVj4YO','12NpEYi7qfIzzHzItlrw1hFb3DB22N2DQ','15GWk7s87nQGlEBCxe3PU7lxYJsXG1rQY','1miE6MBbwkCme0HX1JF08uDkBDjot-DVZ','1pF9TmGrcK56reCN6zRF4xzIaxtZqD6sk']
    credentials,_=google.auth.default(); service=build('drive','v3',credentials=credentials,cache_discovery=False)
    remote_parts=[]; part_root=WORK/'defectoscafeverde-drive-parts'; part_root.mkdir(exist_ok=True)
    for index,file_id in enumerate(chunk_ids):
        part=part_root/f'{ARCHIVE_NAME}.chunk-{index:03d}'; remote_parts.append(part); expected_size=67108864 if index<10 else 15386112
        if part.is_file() and part.stat().st_size==expected_size: continue
        if part.exists(): part.unlink()
        request=service.files().get_media(fileId=file_id,supportsAllDrives=True)
        with part.open('wb') as stream:
            downloader=MediaIoBaseDownload(stream,request,chunksize=16*1024*1024); done=False
            while not done: _,done=downloader.next_chunk()
        print(f'DOWNLOAD BUNDLE: {index+1}/{len(chunk_ids)}',flush=True)
    ARCHIVE=WORK/ARCHIVE_NAME; ARCHIVE_SOURCE='11 Drive API chunks'
    with ARCHIVE.open('wb') as target:
        for part in remote_parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if ARCHIVE.stat().st_size!=EXPECTED_ARCHIVE_BYTES or sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError('Bundle Drive API tidak valid')
project_candidates=[p for p in [Path('/content/drive/MyDrive/Coffee_Bean_Detection'),Path('/content/drive/.shortcut-targets-by-id/19430gSQnXwxodRUvcPZmXLwpy9Z_h_2c/Coffee_Bean_Detection')] if p.is_dir()]
if ARCHIVE_ANCHOR is not None:
    project_candidates += [p for p in ARCHIVE_ANCHOR.parents if p.name=='Coffee_Bean_Detection']
project_candidates=list(dict.fromkeys(p.resolve() for p in project_candidates))
if len(project_candidates)!=1: raise FileNotFoundError(f'Folder proyek bersama harus terlihat tepat satu kali; ditemukan {project_candidates}. Tambahkan shortcut Coffee_Bean_Detection ke My Drive.')
PROJECT=project_candidates[0]
d0_relative=Path('experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json')
d0_candidates=sorted({p for root in drive_roots for p in root.rglob(d0_relative.name) if p.as_posix().endswith(d0_relative.as_posix())})
D0=d0_candidates[0] if len(d0_candidates)==1 else None
if len(d0_candidates)>1: raise RuntimeError(f'Referensi D0 ambigu: {d0_candidates}')
D0_HEADLINE={'macro_map50_95':0.9161970483026481,'bottom3_class_map50_95':0.867287441822945,'worst_class_map50_95':0.8452062524370362}
if D0 is None:
    print('D0 tidak terlihat lewat mount; mencari JSON valid melalui Drive API.',flush=True)
    if 'service' not in globals():
        from google.colab import auth
        auth.authenticate_user()
        import google.auth
        from googleapiclient.discovery import build
        credentials,_=google.auth.default(); service=build('drive','v3',credentials=credentials,cache_discovery=False)
    response=service.files().list(q="name='D0DIRECT_seed42_result.json' and trashed=false",spaces='drive',pageSize=100,includeItemsFromAllDrives=True,supportsAllDrives=True,fields='files(id,name,modifiedTime)').execute()
    valid={}
    for row in response.get('files',[]):
        try:
            content=service.files().get_media(fileId=row['id'],supportsAllDrives=True).execute(); payload=json.loads(content)
            metrics=payload.get('metrics',{})
            exact=all(abs(float(metrics.get(k,-99))-v)<1e-12 for k,v in D0_HEADLINE.items())
            if payload.get('protocol')=='defectoscafeverde-grouped-dcwcf-direct-seed42-v1' and payload.get('arm')=='D0DIRECT' and payload.get('test_images_accessed') is False and exact: valid[hashlib.sha256(content).hexdigest()]=content
        except Exception: pass
    if len(valid)==1:
        D0=WORK/'D0DIRECT_seed42_result.json'; D0.write_bytes(next(iter(valid.values())))
    elif len(valid)>1: raise RuntimeError('Ditemukan beberapa D0 valid tetapi isinya berbeda')
if D0 is None: print('PERINGATAN: raw D0 tidak tersedia. Training RATF1 tetap valid; keputusan formal ditunda.',flush=True)
print('PROJECT:',PROJECT); print('ARCHIVE VALID:',ARCHIVE,'| SOURCE:',ARCHIVE_SOURCE); print('D0:',D0)

In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab sebelum training')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed_files={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first in {'train','val'} or relative in allowed_files): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE - STOP')
import yaml
payload=yaml.safe_load((DATA/'data.yaml').read_text()); payload.pop('test',None)
payload['path']=str(DATA); payload['train']='train/images'; payload['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
AUDIT=DATA/'grouped_audit.json'
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',len(list((DATA/'val/images').glob('*'))),'TEST:',(DATA/'test').exists())

In [ ]:
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=(REPO/'yolo26n.pt').resolve()
OUT=PROJECT/'experiments/defectoscafeverde-ratf1-v1'; OUT.mkdir(parents=True,exist_ok=True)
d0=json.loads(D0.read_text()) if D0 is not None else None
if d0 is not None and (d0.get('protocol')!='defectoscafeverde-grouped-dcwcf-direct-seed42-v1' or d0.get('arm')!='D0DIRECT' or d0.get('test_images_accessed') is not False): raise RuntimeError('D0DIRECT reference tidak valid')
from coffee_detector.experiments.run_defectoscafeverde_ratf import run_static_preflight
from coffee_detector.experiments.run_defectoscafeverde_af2_direct import validate_development_dataset
contract=validate_development_dataset(DATA,AUDIT)
STATIC=OUT/'static_preflight_RATF1_notebook.json'
static=run_static_preflight(PRETRAINED,STATIC,seed=42)
print('D0 REFERENCE:',d0['metrics'] if d0 is not None else D0_HEADLINE)
print('PARAMETERS:',{'native':static['native_parameters'],'candidate':static['candidate_parameters'],'added':static['candidate_added_parameters']})
print('ORTHOGONALITY ERROR:',static['orthogonality_relative_error'])
print('DATA GATES:',contract['gates']); print('STATIC GATES:',static['gates']); print('DECISION:',static['decision'])
assert static['decision']=='PASS','STOP: static audit gagal; training tidak dijalankan.'

In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_ratf','--data-root',str(DATA),'--grouped-audit',str(AUDIT),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))
print('last.pt tersimpan di Drive setiap epoch. Test tidak diekstrak.')

In [ ]:
candidate=json.loads(RESULT.read_text())
if D0 is not None:
    from coffee_detector.experiments.run_defectoscafeverde_ratf import build_decision
    decision=build_decision(D0,RESULT,OUT/'defectoscafeverde_ratf1_seed42_decision.json')
    print('VALUES:',decision['values']); print('DELTAS:',decision['deltas'])
    print('HARD CLASS DELTAS:',decision['hard_class_deltas']); print('PAIRED:',decision['paired']); print('SCREEN:',decision['screen'])
else:
    deltas={k:float(candidate['metrics'][k])-v for k,v in D0_HEADLINE.items()}
    print('STATUS: PROVISIONAL_MISSING_RAW_D0_REFERENCE')
    print('VALUES:',{'D0DIRECT':D0_HEADLINE,'RATF1':candidate['metrics']}); print('DELTAS:',deltas)
    print('RATF1 HARD CLASS VALUES:',candidate['hard_class_map50_95'])
print('TEST:',False)
print('Kirim seluruh output cell ini. Jangan membuka test.')

In [ ]:
if D0 is None: raise FileNotFoundError('Audit partido memerlukan raw D0 result; training RATF1 tetap sah, tetapi audit ditunda sampai D0 dipulihkan.')
D0_CHECKPOINT=PROJECT/'experiments/defectoscafeverde-cwcf-direct-v1/D0DIRECT/D0DIRECT_seed42/weights/best.pt'
RATF_CHECKPOINT=OUT/'RATF1/RATF1_seed42/weights/best.pt'
for label,path in {'D0 checkpoint':D0_CHECKPOINT,'RATF1 checkpoint':RATF_CHECKPOINT}.items():
    if not path.is_file(): raise FileNotFoundError(f'{label} tidak ditemukan: {path}')
PARTIDO_AUDIT=OUT/'ratf1_partido_root_cause.json'
command=[sys.executable,'-u','-m','coffee_detector.analysis.defectoscafeverde_ratf_partido_audit','--data-root',str(DATA),'--d0-result',str(D0),'--ratf-result',str(RESULT),'--d0-checkpoint',str(D0_CHECKPOINT),'--ratf-checkpoint',str(RATF_CHECKPOINT),'--output',str(PARTIDO_AUDIT),'--device','0']
print('MENJALANKAN VALIDATION-ONLY PARTIDO ROOT-CAUSE AUDIT',flush=True)
subprocess.run(command,cwd=REPO,check=True)
audit=json.loads(PARTIDO_AUDIT.read_text())
from IPython.display import display
import pandas as pd
rows=[]
for stage,models in audit['stages'].items():
    for model,values in models.items():
        rows.append({'stage':stage,'model':model,**{key:values[key] for key in ('targets','accessible','matched','correct_class','wrong_class','proposal_accessibility','matched_recall','correct_decision_recall','localization_conditioned_class_accuracy')},'wrong_destinations':values['wrong_destinations']})
display(pd.DataFrame(rows).style.format({key:'{:.2%}' for key in ('proposal_accessibility','matched_recall','correct_decision_recall','localization_conditioned_class_accuracy')}))
score_rows=[{'model':model,**values} for model,values in audit['localized_score_diagnostic'].items()]
display(pd.DataFrame(score_rows).style.format({'mean_localized_margin':'{:+.4f}','median_localized_margin':'{:+.4f}','localized_top1_rate':'{:.2%}','mean_true_class_rank':'{:.2f}'}))
print('RAW LOCALIZATION ACTIVE-ZERO IDENTICAL:',audit['raw_localization_active_zero_identical'])
print('ATTRIBUTION:',audit['attribution'])
print('TRAINING:',audit['training_executed'],'| TEST:',audit['test_images_accessed'])
print('SUMMARY:',PARTIDO_AUDIT)
print('Kirim dua tabel, RAW LOCALIZATION, dan ATTRIBUTION. Jangan training atau membuka test.')